# Imitation and RL summary

Same Panda reach task: load closed-loop metrics from imitation (RGB BC) and RL (state PPO/SAC),
then write `results/metrics/imitation_vs_rl.json`.

Checkpoints use run names `ppo` / `sac` under `models/rl/checkpoints/`.
Modalities differ on purpose (vision vs control). Run after `imitation/eval.ipynb` and `rl/eval.ipynb`.


# Setup

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    repo = Path("/content/drive/MyDrive/visual-policy-learning")
except ImportError:
    here = Path.cwd().resolve()
    repo = next((p for p in [here, *here.parents] if (p / "pyproject.toml").exists()), here)

os.chdir(repo)
if Path("/content").exists():
    os.environ.setdefault("MUJOCO_GL", "egl")
print("cwd:", os.getcwd())

import json


# Load metrics

In [2]:
with open("results/metrics/imitation/eval_results.json") as f:
    imitation = json.load(f)

with open("results/metrics/rl/ppo_vs_sac.json") as f:
    rl = json.load(f)

print("Imitation (RGB BC)")
print(json.dumps(imitation, indent=2))
print("\nRL (privileged state)")
print(json.dumps(rl, indent=2))

Imitation (RGB BC)
{
  "resnet": {
    "success_rate": 0.8,
    "avg_reward": -14.722587195362559,
    "avg_steps": 56.2,
    "reward_std": 9.523183158443718
  },
  "dino": {
    "success_rate": 0.6,
    "avg_reward": -20.119211755276933,
    "avg_steps": 69.8,
    "reward_std": 11.026841027122975
  },
  "clip": {
    "success_rate": 1.0,
    "avg_reward": -10.590704487208846,
    "avg_steps": 43.0,
    "reward_std": 2.8699775922077593
  }
}

RL (privileged state)
{
  "protocol": {
    "ppo_timesteps": 150000,
    "sac_timesteps": 50000,
    "eval_episodes": 25,
    "note": "PPO/SAC and SB3 under the same budgets."
  },
  "grouped": {
    "ppo": {
      "success_rate_mean": 0.96,
      "success_rate_std": 0.0,
      "avg_reward_mean": -13.066318851267907,
      "avg_reward_std": 0.0,
      "avg_steps_mean": 50.0,
      "num_seeds": 1
    },
    "sac": {
      "success_rate_mean": 1.0,
      "success_rate_std": 0.0,
      "avg_reward_mean": -9.22702040385014,
      "avg_reward_std": 0.0

# Comparison table

In [3]:
import pandas as pd

rows = []

for name, m in imitation.items():
    rows.append({
        "family": "imitation",
        "method": name,
        "obs": "RGB",
        "success_rate": m.get("success_rate"),
        "avg_reward": m.get("avg_reward"),
        "avg_steps": m.get("avg_steps"),
    })

for name, m in rl.items():
    # skip metadata blocks like "protocol"
    if not isinstance(m, dict) or "success_rate_mean" not in m:
        continue
    rows.append({
        "family": "rl",
        "method": name,
        "obs": "state",
        "success_rate": m.get("success_rate_mean"),
        "avg_reward": m.get("avg_reward_mean"),
        "avg_steps": m.get("avg_steps_mean"),
        "total_timesteps": m.get("total_timesteps"),
    })

df = pd.DataFrame(rows)
display(df)

rl_metrics = {
    k: v for k, v in rl.items()
    if isinstance(v, dict) and "success_rate_mean" in v
}

comparison = {
    "note": (
        "Same task; imitation uses RGB BC, RL uses privileged state. "
        "Modalities differ on purpose (RGB vs state)."
    ),
    "imitation_rgb": imitation,
    "rl_state": rl_metrics,
    "protocol": rl.get("protocol"),
}

os.makedirs("results/metrics", exist_ok=True)
with open("results/metrics/imitation_vs_rl.json", "w") as f:
    json.dump(comparison, f, indent=4)

print("Wrote results/metrics/imitation_vs_rl.json")


,family,method,obs,success_rate,avg_reward,avg_steps,total_timesteps
0,imitation,resnet,RGB,0.80,-14.722587,56.2,NaN
1,imitation,dino,RGB,0.60,-20.119212,69.8,NaN
2,imitation,clip,RGB,1.00,-10.590704,43.0,NaN
3,rl,ppo,state,0.96,-13.066319,50.0,150000.0
4,rl,sac,state,1.00,-9.227020,42.6,50000.0


Wrote results/metrics/imitation_vs_rl.json
